In [1]:


from langchain_text_splitters import MarkdownHeaderTextSplitter


with open("./resources/评估.md","r",encoding="utf-8") as f:
    doc = f.read()

#切分文档
headers_to_split_on = [
    ("#","H1"),
    ("##","H2"),
    ("###","H3")
]

markdown_splitter = MarkdownHeaderTextSplitter(
    headers_to_split_on=headers_to_split_on,
    strip_headers=False
)
doc = markdown_splitter.split_text(doc)

ids = []
for i,document in enumerate(doc):
    document.id = f"doc_{i}"
    ids.append(document.id)
    # print(document.model_dump_json(indent=2))

In [5]:
from langchain_community.embeddings import DashScopeEmbeddings
from langchain_core.vectorstores import InMemoryVectorStore
from langchain_ollama import OllamaEmbeddings
from dotenv import load_dotenv
load_dotenv()
#初始化向量嵌入模型
# ollam_emb_model = OllamaEmbeddings(
#     model="qwen3-embedding:0.6b",
#     dimensions=1024
# )
emb_model = DashScopeEmbeddings(
    model="text-embedding-v4"
)
#初始化VectoreStore
vectore_store = InMemoryVectorStore(emb_model)

#存入数据库
vectore_store.add_documents(doc)

['doc_0',
 'doc_1',
 'doc_2',
 'doc_3',
 'doc_4',
 'doc_5',
 'doc_6',
 'doc_7',
 'doc_8',
 'doc_9',
 'doc_10',
 'doc_11',
 'doc_12',
 'doc_13',
 'doc_14',
 'doc_15',
 'doc_16',
 'doc_17',
 'doc_18',
 'doc_19',
 'doc_20']

In [12]:
from langchain.chat_models import init_chat_model
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage
from langchain.agents import create_agent
import os
model = init_chat_model(model="deepseek-v4.1-flash-1017",base_url=os.getenv("BASE_URL"))
@tool
def rag_tool(query:str)->str:
    """检索知识库，获取知识片段"""
    documents = vectore_store.similarity_search(query)
    context = "\n".join([doc.page_content for doc in documents])
    return context

myagent = create_agent(model,tools=[rag_tool],system_prompt="""
你是一个教资考试专家，你调用工具检索文档回答用户问题，文档中没有的不要编造，直接说不知道。
    回答直接引用文档中原文，不要自己发挥！
""")
query = "论语中教育的目的是什么？"
response = myagent.invoke({"messages":[HumanMessage(query)]})
for message in response["messages"]:
    message.pretty_print()

================================ Human Message =================================

论语中教育的目的是什么？
================================== Ai Message ==================================
Tool Calls:
  rag_tool (call_0968607ad4af498382e55210)
 Call ID: call_0968607ad4af498382e55210
  Args:
    query: 论语 教育目的
  rag_tool (call_a639ab491554413885a06009)
 Call ID: call_a639ab491554413885a06009
  Args:
    query: 孔子 教育目的 学而优则仕
================================= Tool Message =================================
Name: rag_tool

### （三）孔子及《论语》主要思想  
**教育作用**：庶、富、教；性相近，习相远。  
**教育对象**：“有教无类”，教育民主思想。  
**教育目的**：以完善人格为教育的首要目的，培养士和君子。  
**教学内容**：文、行、忠、义。  
**教学过程**：学、思、习、行。  
**教学原则**：因材施教、启发诱导、学思结合、谦虚笃实。  
**教师观**：其身正，不令而行；其身不正，虽令不从。
### （四）孟子主要思想  
**人性论**：人性本善，人先天具有仁、义、礼、智四个“善端”。  
**教育作用**：发扬善端，培养道德完人，得天下英才而教育之。  
**教学原则**：循序渐进，专心有恒。
### （二）《学记》主要思想  
**教育作用**：化民成俗，其必由学；建国君民，教学为先。  
**教学相长**：教和学两方面互相影响和促进，都得到提高。  
**豫时孙摩**：包括预防性原则、及时性原则、循序渐进原则、集体教育原则。  
**长善救失**：“学者有四失，教者必知之。人之学也，或失则多，或失则寡，或失则易，或失则止。此四者，心之莫同也